In [1]:
import numpy as np
import cv2 as cv

def get_descriptor(img, i, j, win=5):
    """Extrae un patch centrado en (j, i) de tamaño win x win"""
    r = win // 2
    patch = img[j-r:j+r+1, i-r:i+r+1]
    if patch.shape != (win, win):
        return None  # bordes
    return patch.astype(np.float32)

def score_feature_match(f1, f2):
    """SSD entre dos patches"""
    return np.sum((f1 - f2)**2)

def compute_depth(I1, I2, delta_x, f, p, win=5):
    h, w = I1.shape
    depth = np.zeros((h, w), dtype=np.float32)

    for j in range(h):
        for i in range(w):

            f_ij = get_descriptor(I1, i, j, win)
            if f_ij is None:
                continue

            best_score = np.inf
            best_i = None

            for i2 in range(w):  # búsqueda en toda la fila
                f_i2 = get_descriptor(I2, i2, j, win)
                if f_i2 is None:
                    continue

                score = score_feature_match(f_ij, f_i2)
                if score < best_score:
                    best_score = score
                    best_i = i2

            if best_i is not None and (i - best_i) != 0:
                disparity = (i - best_i)
                depth[j, i] = (f * p * delta_x) / disparity

    return depth


In [ ]:
I1 = cv.imread("imagenes/im0.png", cv.IMREAD_GRAYSCALE)
I2 = cv.imread("imagenes/im1.png", cv.IMREAD_GRAYSCALE)

depth_map = compute_depth(I1, I2, delta_x=0.1, f=500, p=0.01)
plt.imshow(depth_map, cmap='inferno')
plt.colorbar()
plt.show()
